**0. Дайте відповідь на запитання:**
1. У розділі 2.1 демо тренер BPE отримує список окремих віршів, а не один склеєний рядок. Що б змінилось (і що б зламалось), якби ми передали один суцільний рядок?
2. З який етапів складається токенізація, поясніть сутність кожного з етапів (див. [туторіал по токенізації](https://huggingface.co/learn/llm-course/en/chapter6/8))
2. Що таке `"<unk>", "<eos>"` в 2.1.?
3. Зменшіть `VOCAB_SIZE` (2.1) удвічі-втричі (наприклад з 16000 до 4000-6000) і перетренуйте BPE-токенізатор на тому самому корпусі. Порівняйте середню кількість токенів на вірш і частку `<unk>` до/після зміни. Поясніть, чому менший словник не підвищує `<unk>`, але впливає на довжину закодованих послідовностей (а отже — і на кількість кроків, які "бачить" модель за той самий `CONTEXT_LEN`).
4. У розділі 2.2 між віршами вставлено токен `<eos>`. Навіщо? Що конкретно піде не так під час тренування mini-GPT, якщо цей роздільник прибрати?
5. Який формат мають і що репрезентують елементи тензорів `train_data` & `val_data` у розділі 2.3?
6. Допишіть в 3.2. `assert` тест, який перевіряє відповідність частин випадкового батчу.
7. У 3.3 `causal_mask` — нижньотрикутна матриця одиниць. Чому саме нижньотрикутна, а не верхньотрикутна? Що б змінилося в поведінці моделі при генерації тексту, якби маски не було?
8. У 3.10 генерація бере чекпоінт із найкращим `val_loss`, а не останній. За яких умов ці два чекпоінти збігаються, а за яких — ні? Подивіться на свою криву навчання (3.9) і скажіть, який випадок у вас.
10. Зменшіть `EVAL_INTERVAL` у 5 разів (наприклад з 250 до 50) і повторно запустіть тренування. Чи стала крива навчання (3.9) помітно гладкішою/інформативнішою? Скільки додаткового часу це коштує (частіші виклики `estimate_loss()` на `EVAL_ITERS` батчах), і чи є сенс робити `EVAL_INTERVAL` ще меншим?
11. У 4.2 той самий промпт дає дуже різний текст при `temperature=0.2` і `temperature=1.4`. Поясніть математично, що робить `temperature` з логітами перед `softmax`, і чому при дуже високій температурі текст стає майже випадковим.
12. За що саме відповідає параметр `top_p`?
13. У 4.2 змініть налаштування `enable_thinking=False` на `True` запустіть повторно клітинку і прокоментуйте зміни у результаті виконання.

### 0.2. Мої відповіді:
1. Якщо передати суцільний рядок:** Токенізатор BPE міг би об'єднати кінець одного вірша з початком іншого. Утворилися б безглузді токени, які існують лише на стику різних документів, що порушило б логіку розбиття слів.

2. Етапи токенізації:
- *Normalization (Нормалізація):* очищення тексту, видалення зайвих пробілів, переведення в нижній регістр.
   - *Pre-tokenization (Пре-токенізація):* грубе розбиття на слова (за пробілами та пунктуацією).
   - *Model (Модель):* застосування алгоритму (BPE, WordPiece) для поділу слів на суб-слова (токени) за словником.
   - *Post-processing (Постобробка):* додавання спеціальних токенів (наприклад, `[CLS]`, `[SEP]`, `<eos>`).
3. **`<unk>` та `<eos>`:** `<unk>` (unknown) — невідомий токен якщо символу немає в словнику. `<eos>` (end of sequence) — токен кінця послідовності, який відділяє один незалежний текст від іншого.
4. **Зменшення `VOCAB_SIZE`:** Частка `<unk>` не зростає, бо базові символи/байти завжди залишаються в словнику. Проте слова розбиваються на дрібніші шматки. Це означає, що при фіксованому `CONTEXT_LEN` (наприклад, 64) мо
5. **Навіщо `<eos>`:** Без нього під час тренування модель намагалася б передбачити перше слово нового вірша, спираючись на останнє слово попереднього, створюючи хибні смислові зв'язки.
6. **Формат тензорів:** Одновимірні тензори (`torch.int64`), де кожне число — це ідентифікатор токена у словнику.
7. **Тест батча:** додаємо в `train` одразу після створення `xb` і `yb`:

`assert xb.shape == yb.shape == (32, context_len)`

`assert (xb[:, 1:] == yb[:, :-1]).all()`

8. **Нижньотрикутна маска:** Щоб на кроці $t$ модель бачила лише поточний і попередні токени ($\le t$). Якби маски не було, модель би "списувала" (заглядала в майбутнє на наступне слово), і навчання б не відбулося.
9. **Чекпоінти:** Вони збігаються, якщо модель не перенавчається (val_loss постійно падає до самого кінця). Не збігаються, якщо починається перенавчання (train_loss падає, а val_loss починає рости) — тоді кращий чекпоінт зберігається раніше за останню ітерацію.
10. **Зменшення `EVAL_INTERVAL`:** Крива стає більш детальною і гладкою, але тренування сильно сповільнюється, бо модель частіше переривається на підрахунок метрик по всьому валідаційному набору. Робити його ще меншим недоцільно — витратимо більше часу на валідацію, ніж на саме навчання.
11. **Математика Temperature:** Логіти діляться на температуру перед `softmax` ($z_i / T$). При $T \to \infty$ розподіл ймовірностей стає майже рівномірним, тому вибір наступного слова стає випадковим. При $T \to 0$ ймовірність найбільшого логіта прямує до 1 (жадібний пошук).
12. **`top_p` (Nucleus sampling):** Відкидає слова з найменшою ймовірністю. Модель вибирає лише з тих топових токенів, сума ймовірностей яких досягає значення $p$ (наприклад, 0.9).
13. **`enable_thinking=True`:** Змушує спеціалізовані моделі генерувати "ланцюжок думок" (Chain-of-Thought) перед формуванням остаточної відповіді.


## 1: Вибір та підготовка датасету
Для виконання лабораторної роботи було обрано україномовний набір даних youscan/ukrainian_sentiment, що містить відгуки користувачів. Вибір обґрунтовано наявністю чіткої емоційної забарвленості текстів та їхньою наближеністю до розмовної мови. Датасет було завантажено, матеріалізовано у структуру DataFrame та очищено від порожніх значень для подальшої коректної токенізації.

In [3]:
import torch
from datasets import load_dataset
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Завантажуємо датасет з українськими новинами
dataset = load_dataset("Zarakun/ukrainian_news", split="train")

# Автоматично беремо 5000 статей
max_rows = min(5000, len(dataset))
dataset = dataset.shuffle(seed=42).select(range(max_rows))
df = dataset.to_pandas()

print("Колонки у датасеті:", df.columns.tolist())

# Витягуємо текст новин
texts_list = df['text'].dropna().tolist()

# Щоб не перевантажити оперативну пам'ять, беремо перші 800 символів кожної новини
texts_list = [text[:800] for text in texts_list]

# Склеюємо у єдиний корпус
corpus_text = "\n\n<eos>\n\n".join(texts_list)

print(f"Кількість текстів: {len(texts_list)}")
print("Розмір корпусу:", len(corpus_text), "символів")
print("--- Приклад тексту ---\n", corpus_text[:500])

Device: cuda


README.md:   0%|          | 0.00/555 [00:00<?, ?B/s]

data/train/train_dataset.parquet: reconstructing file:   0%|          |  0.00B / 3.45MB            

data/train/train_dataset.parquet: downloading bytes:           |  0.00B            

data/test/test_dataset.parquet: reconstructing file:   0%|          |  0.00B /  394kB            

data/test/test_dataset.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1727 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/192 [00:00<?, ? examples/s]

Колонки у датасеті: ['filename', 'language', 'main_topic', 'other_topic', 'text']
Кількість текстів: 1727
Розмір корпусу: 1341669 символів
--- Приклад тексту ---
 У деяких випадках українським пенсіонерам можуть скасувати нарахування пенсії. Однак, уникнути цієї ситуації можливо.Причини скасування пенсій. Пенсійні виплати можуть бути тимчасово припинені або ж скасовані зовсім. Імовірні причини цього зафіксовані у чинному законодавстві.Такими підставами є: смерть пенсіонера; виїзд за кордон на ПМП; виявлення недостовірних документів, які надавалися при оформленні пенсії; відсутність дій на рахунках пенсіонера протягом пів року, йдеться і про неотримання ко


## 2. Токенізатори: BPE, WordPiece, Unigram
Проведено навчання трьох алгоритмів (BPE, WordPiece, Unigram) на новинному корпусі з жорстко обмеженим розміром словника (vocab_size = 500). Специфічні новинні терміни (наприклад, "інфраструктура") розбиваються на дуже дрібні фрагменти. Unigram намагається зберегти більші морфеми, тоді як BPE та WordPiece через малий словник фрагментують слова майже до літер. Для реальної мовної моделі словник має бути значно більшим (від 2000 токенів).

In [4]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

VOCAB_SIZE_COMPARE = 500
SPECIAL_TOKENS = ["<unk>", "<eos>"]

bpe_tok = Tokenizer(models.BPE(unk_token="<unk>"))
bpe_tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe_tok.train_from_iterator(texts_list, trainer=trainers.BpeTrainer(vocab_size=VOCAB_SIZE_COMPARE, special_tokens=SPECIAL_TOKENS))

wp_tok = Tokenizer(models.WordPiece(unk_token="<unk>"))
wp_tok.pre_tokenizer = pre_tokenizers.Whitespace()
wp_tok.train_from_iterator(texts_list, trainer=trainers.WordPieceTrainer(vocab_size=VOCAB_SIZE_COMPARE, special_tokens=SPECIAL_TOKENS))

uni_tok = Tokenizer(models.Unigram())
uni_tok.pre_tokenizer = pre_tokenizers.Whitespace()
uni_tok.train_from_iterator(texts_list, trainer=trainers.UnigramTrainer(vocab_size=VOCAB_SIZE_COMPARE, special_tokens=SPECIAL_TOKENS, unk_token="<unk>"))

test_word = "інфраструктура"
print(f"BPE: {bpe_tok.encode(test_word).tokens}")
print(f"WordPiece: {wp_tok.encode(test_word).tokens}")
print(f"Unigram: {uni_tok.encode(test_word).tokens}")

BPE: ['ÑĸÐ½', 'ÑĦ', 'ÑĢ', 'Ð°ÑģÑĤ', 'ÑĢÑĥ', 'ÐºÑĤ', 'ÑĥÑĢ', 'Ð°']
WordPiece: ['ін', '##ф', '##ра', '##ст', '##ру', '##к', '##ту', '##ра']
Unigram: ['і', 'н', 'ф', 'ра', 'стр', 'у', 'к', 'тур', 'а']


## 3. mini-GPT на своїх даних
Реалізовано та навчено mini-GPT на корпусі новин у двох конфігураціях: базовій із 2 шарами, embedding 64 і контекстом 16 та поглибленій із 4 шарами, embedding 128 і контекстом 32

In [8]:
import numpy as np
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

# Створюємо нормальний словник для моделі
VOCAB_SIZE = 2500
SPECIAL_TOKENS = ["<unk>", "<eos>"]
model_tok = Tokenizer(models.BPE(unk_token="<unk>"))
model_tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
model_tok.train_from_iterator(texts_list, trainer=trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS))

encodings = model_tok.encode_batch(texts_list)
stoi = model_tok.get_vocab()
itos = {v: k for k, v in stoi.items()}

token_ids = np.concatenate([np.array(enc.ids + [stoi.get("<eos>", 0)], dtype=np.int32) for enc in encodings])
data = torch.from_numpy(token_ids.astype(np.int64))
train_data, val_data = data[:int(len(data)*0.9)], data[int(len(data)*0.9):]

class AttentionHead(nn.Module):
    def __init__(self, embed_dim, head_dim, context_len, dropout):
        super().__init__()
        self.query = nn.Linear(embed_dim, head_dim, bias=False)
        self.key = nn.Linear(embed_dim, head_dim, bias=False)
        self.value = nn.Linear(embed_dim, head_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("causal_mask", torch.tril(torch.ones(context_len, context_len)))
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.query(x), self.key(x), self.value(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
        scores = scores.masked_fill(self.causal_mask[:T, :T] == 0, float("-inf"))
        weights = self.dropout(F.softmax(scores, dim=-1))
        return weights @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_heads, context_len, dropout):
        super().__init__()
        head_dim = embed_dim // n_heads
        self.heads = nn.ModuleList([AttentionHead(embed_dim, head_dim, context_len, dropout) for _ in range(n_heads)])
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, embed_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(embed_dim, 4 * embed_dim), nn.ReLU(), nn.Linear(4 * embed_dim, embed_dim), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, n_heads, context_len, dropout):
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim, n_heads, context_len, dropout)
        self.feed_forward = FeedForward(embed_dim, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.feed_forward(self.norm2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, n_heads, n_layers, context_len, dropout):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim)
        self.pos = nn.Embedding(context_len, embed_dim)
        self.blocks = nn.Sequential(*[TransformerBlock(embed_dim, n_heads, context_len, dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)
    def forward(self, idx, targets=None):
        x = self.emb(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        x = self.final_norm(self.blocks(x))
        logits = self.lm_head(x)
        loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), targets.view(-1)) if targets is not None else None
        return logits, loss

def train(name, n_layers, embed_dim, context_len):
    print(f"Тренування {name}...")
    model = MiniGPT(len(itos), embed_dim, 4, n_layers, context_len, 0.1).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    for it in range(400):
        ix = torch.randint(len(train_data) - context_len - 1, (32,))
        xb = torch.stack([train_data[i:i+context_len] for i in ix]).to(device)
        yb = torch.stack([train_data[i+1:i+context_len+1] for i in ix]).to(device)

        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if it % 100 == 0 or it == 399:
            print(f"[{name}] Ітерація {it:3d} | Loss: {loss.item():.4f}")

    os.makedirs('models', exist_ok=True)
    torch.save(model.state_dict(), f"models/{name}.pt")
    return model

model_v1 = train("v1 (Базова)", n_layers=2, embed_dim=64, context_len=16)
model_v2 = train("v2 (Глибша)", n_layers=4, embed_dim=128, context_len=32)

Тренування v1 (Базова)...
[v1 (Базова)] Ітерація   0 | Loss: 7.9842
[v1 (Базова)] Ітерація 100 | Loss: 6.9612
[v1 (Базова)] Ітерація 200 | Loss: 6.8619
[v1 (Базова)] Ітерація 300 | Loss: 6.8193
[v1 (Базова)] Ітерація 399 | Loss: 6.6307
Тренування v2 (Глибша)...
[v2 (Глибша)] Ітерація   0 | Loss: 7.9929
[v2 (Глибша)] Ітерація 100 | Loss: 6.8790
[v2 (Глибша)] Ітерація 200 | Loss: 6.3380
[v2 (Глибша)] Ітерація 300 | Loss: 5.8807
[v2 (Глибша)] Ітерація 399 | Loss: 5.5186


## 4. Playground готових моделей
Продемонстровано застосування високорівневого API pipeline для обробки новинних текстів. Багатомовні моделі успішно впоралися з класифікацією тональності новин, визначенням тематики (Zero-shot) та розпізнаванням іменованих сутностей (NER, зокрема локацій та організацій) на власних україномовних прикладах.

In [11]:
import gc
for name in ["model_v1", "model_v2", "train_data", "val_data"]:
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

from transformers import pipeline

print(pipeline("sentiment-analysis", model="lxyuan/distilbert-base-multilingual-cased-sentiments-student")("Економіка країни продовжує стрімко падати через кризу."))
print(pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")("Київське Динамо здобуло перемогу.", candidate_labels=["політика", "спорт", "економіка"]))

fill = pipeline("fill-mask", model="xlm-roberta-base")
print(fill(f"Президент підписав новий {fill.tokenizer.mask_token} про податки.", top_k=1))

print(pipeline("ner", model="Babelscape/wikineural-multilingual-ner", aggregation_strategy="simple")("Кабінет Міністрів України провів засідання у Києві."))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'negative', 'score': 0.4459063708782196}]


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

{'sequence': 'Київське Динамо здобуло перемогу.', 'labels': ['спорт', 'економіка', 'політика'], 'scores': [0.6371814608573914, 0.19594918191432953, 0.16686928272247314]}


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (6 outputs left out)

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'entity_group': 'ORG', 'score': np.float32(0.620095), 'word': 'Кабінет Міністрів України', 'start': 0, 'end': 25}, {'entity_group': 'LOC', 'score': np.float32(0.999405), 'word': 'Києві', 'start': 45, 'end': 50}]


## 5. Порівняння моделей
Виконано порівняння двох переднавчених моделей для аналізу тональності новинних заголовків.
Модель distilbert, оптимізована під бінарну класифікацію, показує значно вищу точність при роботі з публіцистичним стилем. Модель nlptown/bert орієнтована на відгуки клієнтів (1-5 зірок), тому їй важко оцінювати об'єктивні новинні факти як різко "позитивні" чи "негативні", що призводить до падіння точності

In [13]:
!pip install -q evaluate
import evaluate

# Тестовий набір у стилі заголовків новин
test_data = [
    ("Уряд успішно завершив будівництво нової автомагістралі.", "positive"),
    ("Рівень безробіття сягнув історичного максимуму.", "negative"),
    ("Українські спортсмени вибороли золото на змаганнях.", "positive"),
    ("Корупційний скандал у міністерстві призвів до відставок.", "negative"),
    ("Інфляція продовжує знецінювати заощадження громадян.", "negative"),
    ("Нова реформа освіти отримала схвальні відгуки експертів.", "positive")
]
texts = [t[0] for t in test_data]
true_labels = [1 if t[1] == "positive" else 0 for t in test_data]

pipe1 = pipeline("sentiment-analysis", model="lxyuan/distilbert-base-multilingual-cased-sentiments-student")
pipe2 = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")

pred1 = [1 if p['label'] == "positive" else 0 for p in pipe1(texts)]
pred2 = [1 if int(p['label'].split()[0]) >= 4 else 0 for p in pipe2(texts)]

acc = evaluate.load("accuracy")
print(f"Точність DistilBERT: {acc.compute(predictions=pred1, references=true_labels)['accuracy']}")
print(f"Точність NLPTown BERT: {acc.compute(predictions=pred2, references=true_labels)['accuracy']}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Точність DistilBERT: 0.8333333333333334
Точність NLPTown BERT: 0.6666666666666666


## 6. Творчий застосунок
У межах творчого завдання розроблено інтерактивний веб-застосунок «Розумний рубрикатор новин». Програму реалізовано за допомогою gradio з інтеграцією багатомовної моделі mDeBERTa-v3 для Zero-shot класифікації.

In [16]:
import gradio as gr
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

def categorize_news(text, labels):
    if not text: return "Введіть текст новини"
    candidate_labels = [label.strip() for label in labels.split(",")]
    res = classifier(text, candidate_labels=candidate_labels)

    output = "📊 Результати аналізу:\n\n"
    for label, score in zip(res['labels'], res['scores']):
        output += f"🔸 {label.capitalize()}: {score*100:.1f}%\n"
    return output

demo = gr.Interface(
    fn=categorize_news,
    inputs=[
        gr.Textbox(lines=4, label="Текст новини", placeholder="Вставте текст української новини..."),
        gr.Textbox(label="Можливі рубрики (через кому)", value="політика, економіка, спорт, культура, надзвичайні події")
    ],
    outputs=gr.Textbox(label="Визначена тематика"),
    title="📰 Розумний рубрикатор новин",
    description="Додаток використовує Zero-Shot класифікацію для автоматичного визначення рубрики новини без попереднього навчання."
)
demo.launch(share=True)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a79c6d50e5ac865c27.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
